# 00 — Setup and hardware

**Status: UNEXECUTED FIRST DRAFT.** This notebook prepares the complete AHR v2 XL workspace; it does not train a model.

- **Inputs:** this 16-notebook directory, Python 3.12+, internet access, the eight-GPU machine.
- **Outputs:** project `.venv`, `AHR v2 XL` kernel, pinned environment record, `config/ahr_v2_xl.json`, hardware report.
- **Expected runtime:** roughly 5–15 minutes for first dependency downloads, then under two minutes for the bounded GPU probe; network speed determines setup time.
- **Checkpoint location:** no optimizer run here; installation is idempotent, outputs live in `reports/` and `config/`. Later checkpoints use `checkpoints/<run_id>/`.
- **Resume:** reopen this notebook and Run All. Completed dependency installation is reused. Then select the **AHR v2 XL (Python 3.12)** kernel for notebooks 01–15.

Running this notebook installs dependencies and performs short tensor checks on the GPUs. The recorded 450 W caps and unexplained utilization on GPUs 0–2 need review; no power limit, driver, host service or rental setting is changed. Notebook 09 will measure full-model fit. Width/depth here are provisional **full XL** settings, not measured optimal settings.

## 1. Locate the project

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')


## 2. Shared configuration and notebook loader

In [ ]:
from pathlib import Path
import copy, hashlib, json, os, platform, subprocess, sys, tempfile, time

NOTEBOOK_FILES = [
    '00_setup_and_hardware.ipynb', '01_runtime_and_recovery.ipynb',
    '02_arc_environment.ipynb', '03_datasets_and_replay.ipynb',
    '04_encoder_and_world_model.ipynb', '05_lorentz_workspace_and_memory.ipynb',
    '06_worker.ipynb', '07_scout.ipynb', '08_queen.ipynb',
    '09_full_ahr_integration.ipynb', '10_supervised_training.ipynb',
    '11_joint_training.ipynb', '12_baselines_and_ablations.ipynb',
    '13_benchmark_and_generalization.ipynb', '14_inference_and_model_bundle.ipynb',
    '15_play_and_watch.ipynb',
]
DEPENDENCIES = {0: [], 1: [0], 2: [1], 3: [2], 4: [1], 5: [4],
                6: [5], 7: [6], 8: [7], 9: [2, 3, 8], 10: [9],
                11: [10], 12: [11], 13: [12], 14: [13], 15: [14]}

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                    allow_nan=False).encode()).hexdigest()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(prefix=path.name + '.', suffix='.tmp', dir=path.parent)
    try:
        with os.fdopen(fd, 'w') as stream:
            json.dump(data, stream, indent=2, sort_keys=True, allow_nan=False)
            stream.write('\n'); stream.flush(); os.fsync(stream.fileno())
        os.replace(name, path)
        descriptor = os.open(path.parent, os.O_RDONLY)
        try: os.fsync(descriptor)
        finally: os.close(descriptor)
    finally:
        if os.path.exists(name): os.unlink(name)

def default_config():
    return {
        'schema_version': 1, 'architecture': 'AHR-v2-XL',
        'model': {'d_model': 1024, 'n_heads': 16, 'encoder_layers': 8,
                  'world_layers': 6, 'queen_layers': 8, 'energy_hidden': 2048,
                  'energy_layers': 4, 'patch_size': 4, 'grid_size': 64,
                  'n_colors': 16, 'action_dim': 8, 'event_spatial_dim': 3, 'dropout': 0.0},
        'reasoning': {'worker_candidates': 16, 'scout_candidates': 16,
                      'refine_steps': 4, 'max_depth': 3, 'max_model_calls': 128,
                      'memory_capacity': 256, 'history_length': 32,
                      'geometry': 'lorentz', 'queen_mode': 'learned',
                      'scout_enabled': True, 'latent_clip': 8.0},
        'train': {'seed': 42, 'device': 'cuda:0', 'batch_size': 8,
                  'lr': 1e-4, 'weight_decay': 0.01, 'max_steps': 2000,
                  'grad_clip': 1.0, 'checkpoint_seconds': 600,
                  'checkpoint_steps': 200, 'max_wall_seconds': 3600,
                  'precision': 'bf16', 'stage': 'supervised'},
        'data': {'train_games': [], 'dev_games': [], 'heldout_games': [],
                 'split_manifest': 'data/splits.json', 'replay_path': 'data/replay/train.jsonl',
                 'expert_path': 'data/demonstrations/train.jsonl'},
        'environment': {'operation_mode': 'OFFLINE', 'game_id': None,
                        'max_actions': 256, 'seed': 42},
    }

def ensure_layout(root=None):
    root = Path(root or ROOT)
    for name in ('config', 'data/replay', 'data/demonstrations', 'checkpoints',
                 'reports', 'logs', 'bundles'):
        (root / name).mkdir(parents=True, exist_ok=True)
    return root

def load_config(root=None):
    root = Path(root or ROOT)
    cfg = default_config()
    path = root / 'config' / 'ahr_v2_xl.json'
    def merge(dst, src):
        for key, value in src.items():
            if isinstance(value, dict) and isinstance(dst.get(key), dict): merge(dst[key], value)
            else: dst[key] = copy.deepcopy(value)
    if path.exists(): merge(cfg, json.loads(path.read_text()))
    if cfg['model']['d_model'] % cfg['model']['n_heads']:
        raise ValueError('d_model must be divisible by n_heads')
    if cfg['train']['checkpoint_steps'] > 250 or cfg['train']['checkpoint_seconds'] > 900:
        raise ValueError('Checkpoint cadence must meet the remote working agreement')
    return cfg

def source_manifest(root=None):
    root = Path(root or ROOT)
    result = {}
    for name in NOTEBOOK_FILES:
        path = root / 'notebooks' / name
        if not path.exists(): raise FileNotFoundError(path)
        nb = json.loads(path.read_text())
        cells = [{'source': ''.join(c['source']), 'tags': c.get('metadata', {}).get('tags', [])}
                 for c in nb['cells'] if c['cell_type'] == 'code']
        result[name] = canonical_hash(cells)
    return result

def load_exports(ids, namespace):
    # Explicitly executes library cells only; never imports another notebook's Run All.
    loaded = namespace.setdefault('_AHR_LOADED_EXPORTS', {0})
    active = set()
    def visit(idx):
        idx = int(idx)
        if idx in loaded: return
        if idx in active: raise RuntimeError('Circular notebook dependency')
        if idx not in DEPENDENCIES: raise ValueError(f'Unknown notebook id {idx}')
        active.add(idx)
        for dep in DEPENDENCIES[idx]: visit(dep)
        path = Path(namespace['ROOT']) / 'notebooks' / NOTEBOOK_FILES[idx]
        nb = json.loads(path.read_text())
        for number, cell in enumerate(nb['cells']):
            if cell['cell_type'] == 'code' and 'ahr-export' in cell.get('metadata', {}).get('tags', []):
                exec(compile(''.join(cell['source']), f'{path}:cell-{number}', 'exec'), namespace)
        loaded.add(idx); active.remove(idx)
    for idx in ids: visit(idx)


## 3. Create persistent project directories and initial configuration

In [ ]:
ensure_layout(ROOT)
config_path = ROOT / 'config' / 'ahr_v2_xl.json'
if not config_path.exists(): atomic_json(config_path, default_config())
cfg = load_config()
print('Project:', ROOT)
print('Configuration:', config_path)
print('Architecture:', cfg['architecture'], '| width:', cfg['model']['d_model'])
print('No training has started.')


## 4. Install the project kernel

Run All creates `.venv` under the persistent project directory. PyTorch **2.10.0 with CUDA 12.8** is deliberately pinned to an official Blackwell-capable build; it need not match the host's CUDA 13.3 toolkit. ARC toolkit **0.9.9** requires Python 3.12+. No API key is stored here. Installed third-party packages may contain `.py` files; all **user-authored project logic** stays in these notebooks.

Sources: [official PyTorch wheels](https://pytorch.org/get-started/previous-versions/#v2100), [ARC toolkit package](https://pypi.org/project/arc-agi/0.9.9/).

In [ ]:
import subprocess, sys
if sys.version_info < (3, 12):
    raise RuntimeError('Use the installed /venv/main/bin/python (Python 3.12) to open this setup notebook.')
venv_dir = ROOT / '.venv'
project_python = venv_dir / 'bin' / 'python'
if not project_python.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)
subprocess.run([str(project_python), '-m', 'pip', 'install', 'torch==2.10.0',
                '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([str(project_python), '-m', 'pip', 'install', 'arc-agi==0.9.9',
                'numpy>=2,<3', 'matplotlib>=3.9,<4', 'ipywidgets>=8,<9',
                'pillow>=11,<13', 'nbformat>=5.10,<6', 'ipykernel>=6.29,<8',
                'psutil>=6,<8'], check=True)
subprocess.run([str(project_python), '-m', 'ipykernel', 'install', '--user',
                '--name', 'ahr-v2', '--display-name', 'AHR v2 XL (Python 3.12)'], check=True)
freeze = subprocess.check_output([str(project_python), '-m', 'pip', 'freeze'], text=True)
(ROOT / 'config' / 'requirements-resolved.txt').write_text(freeze)
print('Kernel installed. Select AHR v2 XL for notebook 01 onward.')


## 5. Inspect storage, CPU allocation and GPU topology

In [ ]:
checks = {}
for label, command in {
    'filesystems': ['df', '-hT', str(ROOT), '/workspace', '/dev/shm'],
    'workspace_mount': ['findmnt', '-T', '/workspace'],
    'cpu': ['lscpu'],
    'gpu_topology': ['nvidia-smi', 'topo', '-m'],
    'peer_read_capability': ['nvidia-smi', 'topo', '-p2p', 'r'],
    'gpu_baseline': ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.used,power.limit,utilization.gpu', '--format=csv'],
}.items():
    result = subprocess.run(command, capture_output=True, text=True, timeout=30)
    checks[label] = {'returncode': result.returncode, 'stdout': result.stdout, 'stderr': result.stderr}
    print(label, '\n', result.stdout[:12000])
try:
    capabilities = subprocess.run(['vast-capabilities'], capture_output=True, text=True, timeout=30)
    info = json.loads(capabilities.stdout)
    persistent = info.get('instance', {}).get('workspace_is_volume')
    if not isinstance(persistent, bool): persistent = None
except (OSError, subprocess.TimeoutExpired, ValueError, TypeError, AttributeError):
    persistent = None
checks['workspace_is_volume'] = persistent
if persistent is not True:
    print('WARNING: persistence is not confirmed. Recycle/destroy can erase project files; copy final/best checkpoints off-instance before either action.')
atomic_json(ROOT / 'reports' / 'hardware_inventory.json', checks)


## 6. Bounded CUDA correctness probe

This code runs in the installed kernel's Python without requiring a mid-notebook kernel restart. It checks each GPU with small BF16 matrix operations, synchronization and finite outputs. This is **not** a throughput benchmark or an explanation of the idle-utilization anomaly. The next review should compare sustained workloads and GPU-to-GPU transfer speed before assigning eight-card jobs.

In [ ]:
probe_source = r"""
import json, platform, time
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable'
report = {'torch': torch.__version__, 'cuda_build': torch.version.cuda,
          'python': platform.python_version(), 'compiled_archs': torch.cuda.get_arch_list(), 'devices': []}
for index in range(torch.cuda.device_count()):
    with torch.cuda.device(index):
        start = time.perf_counter()
        x = torch.randn(256, 256, device=f'cuda:{index}', dtype=torch.bfloat16)
        y = x @ x.T
        torch.cuda.synchronize()
        assert torch.isfinite(y).all().item(), f'Nonfinite output on GPU {index}'
        properties = torch.cuda.get_device_properties(index)
        report['devices'].append({'index': index, 'name': properties.name,
                                  'memory_bytes': properties.total_memory,
                                  'capability': list(torch.cuda.get_device_capability(index)),
                                  'probe_seconds': time.perf_counter() - start, 'finite': True})
        del x, y
        torch.cuda.empty_cache()
report['passed'] = len(report['devices']) == 8
print(json.dumps(report))
"""
completed = subprocess.run([str(project_python), '-c', probe_source], capture_output=True, text=True, timeout=180, check=True)
probe = json.loads(completed.stdout.strip().splitlines()[-1])
atomic_json(ROOT / 'reports' / 'hardware_probe.json', probe)
print(json.dumps(probe, indent=2))
if not probe['passed']:
    raise RuntimeError('Expected eight visible GPUs. Inspect hardware_probe.json before continuing.')


## 7. Handoff to recovery validation

Select **AHR v2 XL (Python 3.12)**, open **01_runtime_and_recovery.ipynb**, and Run All. Notebook 01 verifies exact optimization continuation on a tiny diagnostic fixture. Notebook 09 separately verifies continuation on the actual full model. These are engineering checks, not ARC scores.

Keep important data under `/workspace/arc-agi-3-ahrv2`; a local persistent volume is not an off-host backup. No training, data generation or benchmark is launched by notebook 00.